In [ ]:
import subprocess
import time
import csv
from datetime import datetime
import re
import pandas as pd
import matplotlib.pyplot as plt
from config import CSV_FILE_PHY


In [ ]:
INTERFACE = 'wlp1s0'
CSV_FILE = CSV_FILE_PHY

In [ ]:
def get_link_info():
    result = subprocess.run(['iw', 'dev', INTERFACE, 'link'], capture_output=True, text=True)
    return result.stdout

def get_station_dump():
    result = subprocess.run(['iw', 'dev', INTERFACE, 'station', 'dump'], capture_output=True, text=True)
    return result.stdout

def get_survey_dump():
    result = subprocess.run(['iw', 'dev', INTERFACE, 'survey', 'dump'], capture_output=True, text=True)
    return result.stdout

def parse_signal(link_info):
    for line in link_info.splitlines():
        if 'signal:' in line:
            return int(line.strip().split()[1])
    return None

def parse_bitrates_and_mcs(link_info):
    tx_bitrate, rx_bitrate = None, None
    tx_mcs, rx_mcs = None, None
    for line in link_info.splitlines():
        if 'tx bitrate:' in line:
            parts = line.strip().split()
            tx_bitrate = float(parts[2])
            match = re.search(r'MCS\s(\d+)', line)
            if match:
                tx_mcs = int(match.group(1))
        if 'rx bitrate:' in line:
            parts = line.strip().split()
            rx_bitrate = float(parts[2])
            match = re.search(r'MCS\s(\d+)', line)
            if match:
                rx_mcs = int(match.group(1))
    return tx_bitrate, tx_mcs, rx_bitrate, rx_mcs

def parse_station_dump(station_info):
    retries, failures, rx_drops = None, None, None
    for line in station_info.splitlines():
        if 'tx retries:' in line:
            retries = int(line.strip().split(':')[1])
        if 'tx failed:' in line:
            failures = int(line.strip().split(':')[1])
        if 'rx drop misc:' in line:
            rx_drops = int(line.strip().split(':')[1])
    return retries, failures, rx_drops

def parse_noise_floor(survey_info):
    noise_floor = None
    for block in survey_info.split('Survey data from')[1:]:
        lines = block.splitlines()
        freq_line = [line for line in lines if 'frequency:' in line]
        if freq_line and '[in use]' in freq_line[0]:
            noise_lines = [line for line in lines if 'noise:' in line]
            if noise_lines:
                noise_floor = int(noise_lines[0].split(':')[1].split()[0])
            break
    return noise_floor


In [ ]:
with open(CSV_FILE, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow([
        'Time', 
        'Signal', 
        'Noise', 
        'SNR'
    ])

print(f"Starting WiFi monitoring. Data will be saved to {CSV_FILE}... Press Ctrl+C to stop.")

try:
    while True:

        timestamp = datetime.now().strftime('%H:%M:%S.%f')[:-3]
        link_info = get_link_info()
        station_info = get_station_dump()
        survey_info = get_survey_dump()
        
        signal = parse_signal(link_info)
        noise = parse_noise_floor(survey_info)
        snr = signal - noise if signal is not None and noise is not None else None
        
        with open(CSV_FILE, mode='a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([
                timestamp, 
                signal,
                noise, 
                snr
            ])
        
        print(f"[{timestamp}] Noise: {noise} dBm | SNR: {snr} dB")

except KeyboardInterrupt:
    print("\nMonitoring stopped by user. Data saved to", CSV_FILE)


In [ ]:
df = pd.read_csv(CSV_FILE)
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv(CSV_FILE)

# Parse time
df['Time'] = pd.to_datetime(df['Time'])

# Keep only relevant columns
df = df[['Time', 'Noise', 'SNR']]

# Compute elapsed time (seconds)
start_time = df['Time'].iloc[0]
df['Elapsed'] = (df['Time'] - start_time).dt.total_seconds()


# Plot SNR (dB)
plt.figure(figsize=(12,6))
plt.plot(df['Elapsed'], df['SNR'], color='green')
plt.title('SNR (dB) Over Time (Filtered)')
plt.xlabel('Time (s)')
plt.ylabel('SNR (dB)')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot Noise (dBm)
plt.figure(figsize=(12,6))
plt.plot(df['Elapsed'], df['Noise'], color='orange')
plt.title('Noise Floor (dBm) Over Time (Filtered)')
plt.xlabel('Time (s)')
plt.ylabel('Noise (dBm)')
plt.grid(True)
plt.tight_layout()
plt.show()
